# Persona - Realistic lip-sync (SadTalker, free GPU)

Photo + voice.mp3 -> photo wala chehra bolta hua video (hoth + sir hilta hai).

**Pehle ek baar:** upar menu **Runtime -> Change runtime type -> T4 GPU -> Save**.

Phir har cell ko upar se neeche ek-ek karke **Play (▶)** dabao. Pehli baar setup ~5-8 min leta hai.

Rule: sirf apni photo ya AI-generated character. Kisi aur ki photo bina permission ke nahi.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU NAHI MILA - Runtime > Change runtime type > T4 GPU chuno, phir ye cell dobara chalao"

## 2. SadTalker install + fixes (sirf pehli baar, ~5 min)

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/OpenTalker/SadTalker.git
%cd /content/SadTalker
# requirements.txt purane Python ke liye pinned hai, isliye zaroori packages bina strict pin ke
!pip -q install "numpy<2" face-alignment==1.3.5 imageio imageio-ffmpeg librosa resampy pydub scipy kornia==0.6.8 tqdm yacs pyyaml joblib scikit-image basicsr==1.4.2 facexlib==0.3.0 gfpgan av safetensors

In [ ]:
# Naye Python/numpy/torchvision pe SadTalker ki 3 jagah tootti hai - yahan theek karte hain
import re, pathlib, importlib.util

def patch(path, old, new):
    p = pathlib.Path(path)
    s = p.read_text()
    if old in s:
        p.write_text(s.replace(old, new)); print("fixed:", path)
    else:
        print("already ok:", path)

patch("src/face3d/util/my_awing_arch.py", "astype(np.float,", "astype(float,")
patch("src/face3d/util/preprocess.py",
      "trans_params = np.array([w0, h0, s, t[0], t[1]])",
      "trans_params = np.array([w0, h0, float(np.squeeze(s)), float(np.squeeze(t[0])), float(np.squeeze(t[1]))])")

spec = importlib.util.find_spec("basicsr")
deg = pathlib.Path(spec.origin).parent / "data" / "degradations.py"
patch(deg, "torchvision.transforms.functional_tensor", "torchvision.transforms.functional")

## 3. Model download (~1.5 GB, sirf pehli baar)

In [ ]:
%cd /content/SadTalker
!bash scripts/download_models.sh > /dev/null 2>&1; ls -la checkpoints gfpgan/weights

## 4. Photo + voice upload

Do files chuno (Ctrl dabake dono ek saath):
- **photo** (jpg/png): saamne se, achhi light, ek hi chehra, muh band, 512px se badi, chehra photo ka kam se kam 1/3 hissa. Photo jitni saaf, video utna realistic
- **voice.mp3**: Persona ke `video\output\coach-...\` folder se

In [ ]:
from google.colab import files
import os, shutil
os.makedirs("/content/in", exist_ok=True)
up = files.upload()
photo = audio = None
for name in up:
    dst = "/content/in/" + name.replace(" ", "_")
    shutil.move(name, dst)
    if name.lower().endswith((".jpg", ".jpeg", ".png", ".webp")): photo = dst
    if name.lower().endswith((".mp3", ".wav", ".m4a")): audio = dst
print("photo:", photo); print("audio:", audio)
assert photo and audio, "Photo aur audio dono chahiye - ye cell dobara chalao"
!ffmpeg -y -loglevel error -i "$audio" -ar 16000 -ac 1 /content/in/voice.wav

## 5. Video banao (HD 512px, ~3-6 min har 30 sec audio ke liye)

Settings (zaroorat ho to badlo, phir cell dobara chalao):
- `STILL = True` -> sir kam hilega, zyada natural. Robotic/jhoolta lage to True hi rakho.
- `FULL_BODY = True` -> poori photo; `False` -> sirf chehra (lip-sync sabse saaf dikhta hai).
- `HD = True` -> 512px model (zyada realistic, dheema). GPU memory error aaye to `False`.
- `EXPRESSION = 1.0` -> chehre ke expression. 1.1-1.3 = zyada lively, 0.8 = shaant.
- `POSE = 0` -> sir hilane ka style (0-45). Alag numbers try karke jo natural lage wo rakho.


In [ ]:
STILL = True
FULL_BODY = True
HD = True
EXPRESSION = 1.0
POSE = 0

%cd /content/SadTalker
pre = "full" if FULL_BODY else "crop"
still = "--still" if STILL else ""
size = 512 if HD else 256
!rm -rf /content/out
!python inference.py --driven_audio /content/in/voice.wav --source_image "$photo" --result_dir /content/out --preprocess $pre $still --enhancer gfpgan --size $size --expression_scale $EXPRESSION --pose_style $POSE

## 6. Download

In [ ]:
import glob
vids = sorted(glob.glob("/content/out/**/*.mp4", recursive=True), key=os.path.getmtime)
assert vids, "Video nahi bana - upar cell 5 ka error copy karke Kiro ko bhejo"
final = "/content/persona_lipsync.mp4"
shutil.copy(vids[-1], final)
from IPython.display import Video, display
display(Video(final, embed=True, width=360))
files.download(final)

### Kuch galat hua?
- **CUDA / GPU error** -> Runtime type T4 GPU hai? Phir `Runtime -> Restart session` aur cell 2 se dobara.
- **"face not detected"** -> photo mein chehra bada aur saamne se ho, chashma/haath na ho.
- **Koi aur error** -> error ki aakhri 15 lines copy karke Kiro ko bhejo.

Colab session band hone pe sab mit jaata hai (photo bhi) - agli baar cell 2 se phir chalana padega.